In [2]:
# ── Config ─────────────────────────────────────────────
INPUT_CSV   = 'results.csv'
OUTPUT_XLSX = 'scorecard.xlsx'

# BIZ days (Mon–Fri, holidays ignored).
ACK_DEADLINE_BD       = 5    # NH RSA 91-A. Past this without acknowledgement      → −5  (No response at all)
FULFILL_DEADLINE_BD   = 25   # Past this acknowledged-but-pending (~5 weeks)        → −3  (Acknowledged but never fulfilled)
IMMEDIATE_BD          = 10    # Completed within this (≤)                            → +5  (Immediate electronic access)
DELAYED_FULFILLED_BD  = 60   # Completed past this (>)                              → +3  (Delayed access after back and forth)
                             # Completed between IMMEDIATE_BD and DELAYED_FULFILLED_BD → +4 (Timely access)

AUTO_EXCLUDE_CATEGORIES = {'RFP List'}
AUTO_EXCLUDE_STATUS_CODES = {'fix'}

JURISDICTION_REMAP = {
    'Berlin School District/SAU 3':                          'Berlin',
    'SAU6, serving the communities of Claremont and Unity':  'Claremont',
    'Concord School District/SAU8':                          'Concord',
    'Dover School District/SAU11':                           'Dover',
    'Franklin School District/SAU18':                        'Franklin',
    'Manchester School District':                            'Manchester',
    'Keene School District/SAU29':                           'Keene',
    'Laconia School District/SAU30':                         'Laconia',
    'Lebanon School District/SAU88':                         'Lebanon',
    'Nashua School District':                                'Nashua',
    'Portsmouth School District':                            'Portsmouth',
    'Rochester School District':                             'Rochester',
    'Somersworth School District/SAU56':                     'Somersworth',
    'Contoocook Valley School District/SAU1':                'Peterborough',
    'Shaker Regional School District/SAU80':                 'Belmont',
    'SAU70':                                                 'Hanover',
    'SAU20':                                                 'Gorham',
    'SAU67':                                                 'Bow',
    'Hopkinton School District':                             'Hopkinton',
    'Derry Cooperative School District':                     'Derry',
    'Salem School District/SAU57':                           'Salem',
    'Monadnock Regional School District':                    'Troy',
    'Oyster River Cooperative School District':              'Durham',
}

AGENCY_REMAP = {
    ('Lebanon', 'City Clerk'):         'Lebanon City Clerk',
    ('Lebanon', 'Lebanon Clerk'):      'Lebanon City Clerk',
    ('Nashua',  'Nashua City Clerk'):  'Nashua City Clerk',
    ('Nashua',  'Nashua Clerk'):       'Nashua City Clerk',
}

In [4]:
import csv, re
from collections import defaultdict
from datetime import date, datetime
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

with open(INPUT_CSV, newline='', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))

print(f'Loaded {len(rows)} requests from {INPUT_CSV}')

Loaded 280 requests from results.csv


In [6]:
_TRAIL_PAREN = re.compile(r'\s*\([^)]*\)\s*$')

def category(title: str) -> str:
    return _TRAIL_PAREN.sub('', title).strip()

def resolved_jurisdiction(row: dict) -> str:
    """Map school-district agencies up to their host city/town."""
    return JURISDICTION_REMAP.get(row['Agency Name'], row['Jurisdiction'])

def resolved_agency(row: dict) -> str:
    j = resolved_jurisdiction(row)
    a = row['Agency Name']
    return AGENCY_REMAP.get((j, a), a)

def normalized_tags(row: dict) -> str:
    return (row.get('Tags') or '').lower().replace('nh-in-person', 'nh-inperson')

def has_tag(row: dict, fragment: str) -> bool:
    return fragment in normalized_tags(row)

def _parse_dt(s: str):
    if not s: return None
    try: return datetime.strptime(s.split()[0], '%Y-%m-%d').date()
    except ValueError: return None

def business_days_between(start: date, end: date) -> int:
    if not start or not end or end <= start: return 0
    total = (end - start).days
    weeks, rem = divmod(total, 7)
    biz = weeks * 5
    start_wd = start.weekday()
    for d in range(1, rem + 1):
        if (start_wd + d) % 7 < 5: biz += 1
    return biz

def business_days_open(row: dict, today: date = None) -> int:
    submitted = _parse_dt(row.get('Date Submitted'))
    done      = _parse_dt(row.get('Date Done'))
    return business_days_between(submitted, done or (today or date.today()))

def auto_score(row: dict, today: date = None):
    """Map a CSV row to a rubric score, or None if it can't be auto-scored.

    Auto mapping (matches index.html rubric labels):
        +5  done within IMMEDIATE_BD                                              Immediate electronic access
        +4  done between IMMEDIATE_BD and DELAYED_FULFILLED_BD                    Timely access
        +3  done past DELAYED_FULFILLED_BD (no in-person/citizen tag)             Delayed access after back and forth
        +2  done + tag nh-inperson|nh-citizen                                     In-person pickup / residency required
        +1  excessive redaction                                                   manual only
         0  no_docs                                                               No responsive documents exist
        −1  process confusion (fix is auto-excluded; -1 reserved for manual use)  manual only
        −2  tag nh-privacy / *privacy*                                            Rejection on records requiring balancing test
        −3  processed/submitted past FULFILL_DEADLINE_BD                          Acknowledged but never fulfilled
        −4  rejected (no privacy tag)                                             Rejected records routinely considered public
        −5  ack past ACK_DEADLINE_BD                                              No response at all
    """
    code = (row.get('Status Code') or '').strip()
    bd   = business_days_open(row, today)

    if has_tag(row, 'privacy'):                                       
        return -2

    if code == 'done':
        if has_tag(row, 'nh-inperson') or has_tag(row, 'nh-citizen'):
            return 2                                                  # In-person pickup / residency required
        if bd > DELAYED_FULFILLED_BD:                                 return 3     # Delayed access after back and forth
        if bd <= IMMEDIATE_BD:                                        return 5     # Immediate electronic access
        return 4                                                                    # Timely access
    if code == 'no_docs':    return 0
    if code == 'fix':        return None                              # auto-excluded; doesn't enter the average
    if code == 'partial':    return None                              # manual
    if code == 'rejected':   return -4                                # blanket rejection on routinely-public records
    if code == 'ack':
        return -5 if bd > ACK_DEADLINE_BD else None
    if code in ('processed', 'submitted'):
        return -3 if bd > FULFILL_DEADLINE_BD else None
    return None

from collections import Counter
for s, n in sorted(Counter(auto_score(r) for r in rows).items(), key=lambda kv: (kv[0] is None, kv[0])):
    print(f'  {str(s):>5}  {n:4d}')

     -5    27
     -4    30
     -3    27
     -2     2
      0    66
      2     8
      3     1
      4    31
      5    43
   None    45


In [7]:
agencies = defaultdict(lambda: defaultdict(list))   # (juris, agency) → cat → [rows]
for r in rows:
    j = resolved_jurisdiction(r)
    a = resolved_agency(r)
    agencies[(j, a)][category(r['Title'])].append(r)

categories  = sorted({category(r['Title']) for r in rows})
agency_keys = sorted(agencies.keys())               # [(juris, agency), …]
jurisdictions = sorted({j for j, _ in agency_keys})

print(f'{len(agency_keys)} agencies across {len(jurisdictions)} jurisdictions')
print(f'{len(categories)} request categories:')
for c in categories:
    flag = '  (excluded by default)' if c in AUTO_EXCLUDE_CATEGORIES else ''
    print(f'  • {c}{flag}')

69 agencies across 23 jurisdictions
11 request categories:
  • AI chat logs
  • AI policy
  • Ethics and Conflict of Interest Rules
  • Highest-value RFPs
  • Leadership and constituent emails
  • PD Chief Performance Evaluation
  • PD Settlement Agreements
  • Parental Bill of Rights
  • Parental Objections
  • Performance Evaluation
  • RFP List  (excluded by default)


In [8]:
# ── Build workbook ─────────────────────────────────────
wb = Workbook()
ws = wb.active
ws.title = 'Agencies'

BOLD       = Font(bold=True)
LINK       = Font(color='0563C1', underline='single')
HDR_FILL   = PatternFill('solid', fgColor='EEF3FA')
BLOCK_FILL = {                                  # subtle column-block tinting
    'status':  PatternFill('solid', fgColor='F7F8FA'),
    'exclude': PatternFill('solid', fgColor='FDF2D6'),
    'auto':    PatternFill('solid', fgColor='EAF3EA'),
    'manual':  PatternFill('solid', fgColor='FAEAEA'),
}

N = len(categories)
headers = ['Jurisdiction', 'Agency', 'Requests']
headers += [f'{c} — Status'  for c in categories]
headers += [f'{c} — Exclude' for c in categories]
headers += [f'{c} — Auto'    for c in categories]
headers += [f'{c} — Manual'  for c in categories]
headers += ['Calculated Avg', 'Manual Avg']
ws.append(headers)

STATUS_START  = 4
EXCLUDE_START = STATUS_START + N
AUTO_START    = EXCLUDE_START + N
MANUAL_START  = AUTO_START + N
CALC_COL      = MANUAL_START + N
MAN_COL       = CALC_COL + 1

for c in ws[1]:
    c.font = BOLD
    c.fill = HDR_FILL
    c.alignment = Alignment(wrap_text=True, vertical='center')

for ridx, (j, a) in enumerate(agency_keys, start=2):
    req_by_cat = agencies[(j, a)]
    total = sum(len(v) for v in req_by_cat.values())

    ws.cell(row=ridx, column=1, value=j)
    ws.cell(row=ridx, column=2, value=a)
    ws.cell(row=ridx, column=3, value=total)

    for i, cat in enumerate(categories):
        col_s, col_e, col_au, col_m = (STATUS_START+i, EXCLUDE_START+i, AUTO_START+i, MANUAL_START+i)
        cs, ce, ca, cm = (ws.cell(row=ridx, column=c) for c in (col_s, col_e, col_au, col_m))
        cs.fill, ce.fill, ca.fill, cm.fill = (
            BLOCK_FILL['status'], BLOCK_FILL['exclude'], BLOCK_FILL['auto'], BLOCK_FILL['manual']
        )

        reqs = req_by_cat.get(cat, [])
        if not reqs:
            cs.value, ce.value = '—', True            # no request → exclude
            continue

        req = sorted(reqs, key=lambda r: r.get('Date Submitted') or '', reverse=True)[0]
        label = req['Status']
        if len(reqs) > 1: label += f' (+{len(reqs)-1} more)'
        cs.value = label
        if req.get('MuckRock URL'):
            cs.hyperlink = req['MuckRock URL']
            cs.font = LINK

        excluded = (
            cat in AUTO_EXCLUDE_CATEGORIES
            or (req.get('Status Code') or '').strip() in AUTO_EXCLUDE_STATUS_CODES
        )
        ce.value = excluded

        au = auto_score(req)
        if au is not None:
            ca.value = au

    auto_r = f'{get_column_letter(AUTO_START)}{ridx}:{get_column_letter(AUTO_START + N - 1)}{ridx}'
    excl_r = f'{get_column_letter(EXCLUDE_START)}{ridx}:{get_column_letter(EXCLUDE_START + N - 1)}{ridx}'
    man_r  = f'{get_column_letter(MANUAL_START)}{ridx}:{get_column_letter(MANUAL_START + N - 1)}{ridx}'
    ws.cell(row=ridx, column=CALC_COL,
            value=f'=IFERROR(AVERAGEIFS({auto_r},{excl_r},FALSE),"")')
    ws.cell(row=ridx, column=MAN_COL,
            value=f'=IFERROR(AVERAGEIFS({man_r},{excl_r},FALSE),"")')

ws.freeze_panes = 'D2'
ws.row_dimensions[1].height = 38
for i, h in enumerate(headers, start=1):
    width = 14
    if i in (1, 2):                         width = 26
    elif STATUS_START <= i < EXCLUDE_START: width = 22
    elif EXCLUDE_START <= i < AUTO_START:   width = 10
    elif AUTO_START <= i < MANUAL_START:    width = 8
    elif MANUAL_START <= i < CALC_COL:      width = 10
    elif i in (CALC_COL, MAN_COL):          width = 14
    ws.column_dimensions[get_column_letter(i)].width = width

In [ ]:
# ── Jurisdictions tab — live cross-sheet averages ──────
ws2 = wb.create_sheet('Jurisdictions')
ws2.append(['Jurisdiction', 'Agencies', 'Calculated Avg', 'Manual Avg', 'Summary'])
for c in ws2[1]:
    c.font = BOLD
    c.fill = HDR_FILL

last_row   = len(agency_keys) + 1
juris_rng  = f'Agencies!$A$2:$A${last_row}'
calc_rng   = f'Agencies!${get_column_letter(CALC_COL)}$2:${get_column_letter(CALC_COL)}${last_row}'
manual_rng = f'Agencies!${get_column_letter(MAN_COL)}$2:${get_column_letter(MAN_COL)}${last_row}'

for ridx, j in enumerate(jurisdictions, start=2):
    ws2.cell(row=ridx, column=1, value=j)
    ws2.cell(row=ridx, column=2, value=f'=COUNTIF({juris_rng},A{ridx})')
    ws2.cell(row=ridx, column=3, value=f'=IFERROR(AVERAGEIFS({calc_rng},{juris_rng},A{ridx}),"")')
    ws2.cell(row=ridx, column=4, value=f'=IFERROR(AVERAGEIFS({manual_rng},{juris_rng},A{ridx}),"")')
    # Column 5 (Summary) left blank for human-authored summary text per jurisdiction.

ws2.freeze_panes = 'B2'
ws2.column_dimensions['A'].width = 26
ws2.column_dimensions['B'].width = 12
ws2.column_dimensions['C'].width = 14
ws2.column_dimensions['D'].width = 14
ws2.column_dimensions['E'].width = 70   # Summary — give it room
for row in ws2.iter_rows(min_row=2):
    row[4].alignment = Alignment(wrap_text=True, vertical='top')

wb.save(OUTPUT_XLSX)
print(f'✓ wrote {OUTPUT_XLSX}: {len(agency_keys)} agencies, {len(jurisdictions)} jurisdictions, {len(categories)} categories')